# Clase 230 — Capstone 2: NLP o series de tiempo end-to-end

Este capstone tiene **dos ramas opt-in**:

- **Rama A — NLP**: clasificación de texto / NER / RAG mini con transformers + FastAPI. Descrita en el README; requiere descargar pesos de Hugging Face.
- **Rama B — Series de tiempo**: forecasting con baselines + ML + intervalos. **Implementada aquí** porque corre 100% offline con numpy/pandas/sklearn (+ statsmodels opcional).

El alumno puede swapear este notebook por uno equivalente de NLP siguiendo el README. Seed = 42, self-contained.

Requiere: `pip install numpy pandas scikit-learn scipy` (+ opcional `statsmodels`).

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import Ridge, QuantileRegressor
from sklearn.metrics import mean_absolute_error
from scipy import stats

rng = np.random.default_rng(42)

# Serie sintetica: 3 anios diarios = 1095 puntos
n_days = 365 * 3
t = np.arange(n_days)
trend   = 100 + 0.05 * t
weekly  = 8 * np.sin(2 * np.pi * t / 7)
yearly  = 15 * np.sin(2 * np.pi * t / 365.25 - np.pi / 2)
noise   = rng.normal(0, 3, n_days)
y = trend + weekly + yearly + noise

dates = pd.date_range('2023-01-01', periods=n_days, freq='D')
df = pd.DataFrame({'date': dates, 'y': y}).set_index('date')
print(f'serie: {df.shape[0]} dias | rango {df.index.min().date()} -> {df.index.max().date()}')
print(df.head())

## 1. EDA temporal: descomposicion manual + ACF

In [ ]:
trend_est    = df['y'].rolling(28, center=True).mean()
detrended    = df['y'] - trend_est
seasonal_est = detrended.groupby(df.index.dayofweek).transform('mean')
resid        = df['y'] - trend_est - seasonal_est

print('Componentes (mean / std):')
for name, s in [('trend', trend_est), ('seasonal', seasonal_est), ('resid', resid)]:
    print(f'  {name:10s} mean={s.mean():7.2f}  std={s.std():6.2f}')

def acf(x, max_lag=30):
    x = np.asarray(x) - np.mean(x)
    var = np.dot(x, x)
    return np.array([np.dot(x[:-k], x[k:]) / var for k in range(1, max_lag + 1)])

acfs = acf(df['y'].values, 30)
print('\nACF top picos (lag, valor):')
for lag in sorted(np.argsort(-np.abs(acfs))[:5]):
    print(f'  lag={lag+1:2d}: {acfs[lag]:+.3f}')

## 2. Split temporal honesto (sin shuffle)

Train hasta T-90, validacion T-90 a T-30, test ultimos 30 dias.

In [ ]:
train = df.iloc[:-90].copy()
val   = df.iloc[-90:-30].copy()
test  = df.iloc[-30:].copy()
print(f'train: {len(train):4d} dias  ({train.index.min().date()} -> {train.index.max().date()})')
print(f'val:   {len(val):4d} dias  ({val.index.min().date()} -> {val.index.max().date()})')
print(f'test:  {len(test):4d} dias  ({test.index.min().date()} -> {test.index.max().date()})')

def smape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return 100 * np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-9))

## 3. Baseline naive: y_hat[t+h] = y[t]

In [ ]:
last_train = train['y'].iloc[-1]
pred_naive = np.full(len(test), last_train)
print(f'naive sMAPE test: {smape(test["y"], pred_naive):.3f}%')
print(f'naive MAE  test: {mean_absolute_error(test["y"], pred_naive):.3f}')

## 4. Baseline seasonal naive (semanal)

In [ ]:
full = pd.concat([train, val])
last_week = full['y'].iloc[-7:].values
pred_snaive = np.tile(last_week, int(np.ceil(len(test) / 7)))[:len(test)]
print(f'seasonal naive sMAPE test: {smape(test["y"], pred_snaive):.3f}%')

## 5. ETS aditivo (statsmodels, opcional)

Si statsmodels no esta disponible, se reemplaza por seasonal naive.

In [ ]:
try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    model_ets = ExponentialSmoothing(
        train['y'], trend='add', seasonal='add', seasonal_periods=7
    ).fit()
    pred_ets = model_ets.forecast(len(val) + len(test))[-len(test):].values
    print(f'ETS sMAPE test: {smape(test["y"], pred_ets):.3f}%')
except Exception as e:
    print(f'statsmodels no disponible o fallo ({e}); usando seasonal naive como sustituto.')
    pred_ets = pred_snaive

## 6. ML approach: features lag + rolling -> Ridge

In [ ]:
def make_features(s, lags=(1, 2, 3, 7, 14), rolls=(7, 28)):
    feats = pd.DataFrame(index=s.index)
    for L in lags:
        feats[f'lag_{L}'] = s.shift(L)
    for w in rolls:
        feats[f'roll_mean_{w}'] = s.shift(1).rolling(w).mean()
        feats[f'roll_std_{w}']  = s.shift(1).rolling(w).std()
    feats['dow'] = feats.index.dayofweek
    feats['doy'] = feats.index.dayofyear
    return feats

X_all = make_features(df['y'])
y_all = df['y']
mask  = X_all.notna().all(axis=1)
X_all, y_all = X_all[mask], y_all[mask]

X_tr = X_all.loc[:train.index[-1]]
y_tr = y_all.loc[:train.index[-1]]
X_te = X_all.loc[test.index[0]:]
y_te = y_all.loc[test.index[0]:]

ridge = Ridge(alpha=1.0).fit(X_tr, y_tr)
pred_ridge = ridge.predict(X_te)
print(f'Ridge sMAPE test: {smape(y_te, pred_ridge):.3f}%')

print('\n--- Resumen ---')
for name, p in [('naive', pred_naive), ('seasonal_naive', pred_snaive), ('ETS', pred_ets), ('Ridge', pred_ridge)]:
    p_aligned = p[:len(y_te)] if len(p) >= len(y_te) else np.concatenate([p, [p[-1]] * (len(y_te) - len(p))])
    print(f'  {name:16s} sMAPE = {smape(y_te, p_aligned):.3f}%')

## 7. Backtesting con expanding window (5 folds)

In [ ]:
horizon = 30
n_folds = 5
smapes = []
for k in range(n_folds):
    cutoff_idx = len(X_all) - (n_folds - k) * horizon
    X_tr_k = X_all.iloc[:cutoff_idx]
    y_tr_k = y_all.iloc[:cutoff_idx]
    X_te_k = X_all.iloc[cutoff_idx:cutoff_idx + horizon]
    y_te_k = y_all.iloc[cutoff_idx:cutoff_idx + horizon]
    m = Ridge(alpha=1.0).fit(X_tr_k, y_tr_k)
    s = smape(y_te_k, m.predict(X_te_k))
    smapes.append(s)
    print(f'  fold {k+1}: train={len(X_tr_k):4d}  test={len(X_te_k):2d}  sMAPE={s:.3f}%')

print(f'\nbacktesting Ridge: mean sMAPE = {np.mean(smapes):.3f}%  +/- {np.std(smapes):.3f}')

## 8. Quantile regression para intervalos P10/P90 + pinball loss

In [ ]:
q10 = QuantileRegressor(quantile=0.10, alpha=0.001, solver='highs').fit(X_tr, y_tr)
q90 = QuantileRegressor(quantile=0.90, alpha=0.001, solver='highs').fit(X_tr, y_tr)

pred_p10 = q10.predict(X_te)
pred_p90 = q90.predict(X_te)

coverage = np.mean((y_te.values >= pred_p10) & (y_te.values <= pred_p90))
width    = np.mean(pred_p90 - pred_p10)
print(f'cobertura empirica P10-P90: {coverage:.1%}  (target 80%)')
print(f'ancho promedio del intervalo: {width:.2f}')

def pinball(y_true, y_pred, tau):
    d = y_true - y_pred
    return np.mean(np.maximum(tau * d, (tau - 1) * d))

print(f'pinball loss tau=0.10: {pinball(y_te.values, pred_p10, 0.10):.3f}')
print(f'pinball loss tau=0.90: {pinball(y_te.values, pred_p90, 0.90):.3f}')

## 9. Plot ASCII: actual vs prediccion Ridge con bandas P10/P90

In [ ]:
lo = min(pred_p10.min(), y_te.min())
hi = max(pred_p90.max(), y_te.max())
width_chars = 50

def scale(v):
    return int((v - lo) / (hi - lo) * (width_chars - 1))

print(f'{"date":12s}  {"actual":>7s}  {"yhat":>7s}  P10---P90')
for i, (d, yt, ypp, p10, p90) in enumerate(zip(
    y_te.index, y_te.values, pred_ridge, pred_p10, pred_p90
)):
    line = [' '] * width_chars
    a, b = scale(p10), scale(p90)
    for k in range(a, b + 1):
        line[k] = '-'
    line[scale(ypp)] = 'o'
    line[scale(yt)]  = '*'
    if i % 3 == 0:
        print(f'{d.date()!s:12s}  {yt:7.2f}  {ypp:7.2f}  {"".join(line)}')
print('\n* = actual   o = prediccion Ridge   --- = banda P10-P90')

## 10. Drift conceptual: residuos 1a vs 2a mitad del test (KS)

In [ ]:
resid_test = y_te.values - pred_ridge
mid = len(resid_test) // 2
r1, r2 = resid_test[:mid], resid_test[mid:]

ks_stat, p_val = stats.ks_2samp(r1, r2)
print(f'residuos 1a mitad: mean={r1.mean():+.3f}  std={r1.std():.3f}')
print(f'residuos 2a mitad: mean={r2.mean():+.3f}  std={r2.std():.3f}')
print(f'KS test: stat={ks_stat:.3f}  p-value={p_val:.3f}')
verdict = 'NO hay evidencia de drift' if p_val > 0.05 else 'DRIFT detectado -- re-entrenar'
print(f'-> {verdict} (alpha=0.05)')

## 11. Stub FastAPI para `/forecast`

In [ ]:
fastapi_stub = '''
# app/main.py
from fastapi import FastAPI
import joblib, pandas as pd

app = FastAPI(title="forecast-service")
model = joblib.load("artifacts/ridge.pkl")
q10   = joblib.load("artifacts/q10.pkl")
q90   = joblib.load("artifacts/q90.pkl")
history = pd.read_parquet("artifacts/history.parquet")

@app.get("/forecast")
def forecast(horizon: int = 14):
    preds, p10s, p90s = [], [], []
    h = history.copy()
    for _ in range(horizon):
        feats = make_features(h["y"]).iloc[[-1]]
        yhat  = float(model.predict(feats)[0])
        preds.append(yhat)
        p10s.append(float(q10.predict(feats)[0]))
        p90s.append(float(q90.predict(feats)[0]))
        next_date = h.index[-1] + pd.Timedelta(days=1)
        h.loc[next_date] = yhat   # recursive forecast
    return {"horizon": horizon, "yhat": preds, "p10": p10s, "p90": p90s}
'''
print(fastapi_stub)

## Checklist de entregables del capstone

- [ ] `README.md` del proyecto: problema, rama (A=NLP / B=series), metrica primaria, resultado.
- [ ] Notebook de EDA + modelado con seed 42 y MLflow tracking visible.
- [ ] Baselines reportados (naive + seasonal naive + ETS o TF-IDF + LogReg).
- [ ] Modelo moderno que **supere** al mejor baseline en la metrica primaria.
- [ ] Backtesting con expanding window (>=5 folds) reportando mean +/- std.
- [ ] Intervalos de prediccion (quantile/conformal en series; calibracion en NLP).
- [ ] Analisis por slice/horizonte: al menos un caso debil identificado.
- [ ] `Dockerfile` + `compose.yml` que levantan FastAPI + MLflow UI.
- [ ] Endpoint funcional (`/predict` o `/forecast`) con latencia <500 ms.
- [ ] Reporte de drift (Evidently o KS sobre residuos).
- [ ] **Model Card** (1 pagina): datos, metricas, sesgos, plan de monitoreo.